# Analyse historique d’une action avec `yfinance`

Ce notebook adapte le script de la vidéo aux versions récentes de `yfinance`.

L’ancienne combinaison `pandas_datareader` + `yf.pdr_override()` a été remplacée par `yf.download()`. La version actuelle de `yfinance` utilise par défaut `auto_adjust=True`, ce qui signifie que la colonne `Close` correspond aux cours ajustés. Le notebook force aussi `multi_level_index=False` pour obtenir des colonnes simples lors du téléchargement d’un seul titre.

> Ce notebook est destiné à l’apprentissage et à la recherche. Les données Yahoo Finance sont soumises aux conditions d’utilisation de Yahoo Finance.

In [ ]:
# Installation à exécuter une seule fois si nécessaire.
# Dans Jupyter, le préfixe % permet d’utiliser le même environnement que le noyau.
%pip install -U yfinance pandas matplotlib

In [ ]:
import datetime as dt
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## Paramètres et téléchargement des données

Avec `auto_adjust=True`, les colonnes OHLC sont ajustées pour les divisions d’actions et les dividendes. Il n’est donc plus nécessaire d’utiliser `Adj Close` : la colonne `Close` est utilisée directement.

In [ ]:
# Entrée utilisateur ; exemple : AAPL, MSFT, TSLA ou MC.PA
stock = input("Entrez le symbole boursier (ou 'quit' pour arrêter) : ").strip().upper()

if stock.lower() == "QUIT":
    raise SystemExit("Arrêt demandé.")

start = dt.datetime(2019, 1, 1)
end = dt.datetime.now()

# API actuelle de yfinance : pas de pdr_override() et pas de pandas_datareader.
df = yf.download(
    stock,
    start=start,
    end=end,
    interval="1d",
    auto_adjust=True,
    actions=False,
    progress=False,
    multi_level_index=False,
)

if df.empty:
    raise ValueError(
        f"Aucune donnée reçue pour {stock!r}. Vérifiez le symbole et votre connexion."
    )

# Sécurité pour les versions/configurations qui renverraient un MultiIndex.
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

required = {"Open", "High", "Low", "Close", "Volume"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Colonnes manquantes dans la réponse Yahoo Finance : {sorted(missing)}")

df.index = pd.to_datetime(df.index)
df = df.sort_index()

print(f"{len(df):,} lignes téléchargées pour {stock}")
display(df.tail())

## Moyenne mobile simple de 50 jours

In [ ]:
ma = 50
sma_column = f"SMA_{ma}"
df[sma_column] = df["Close"].rolling(window=ma, min_periods=ma).mean()

# On retire les premières lignes qui ne possèdent pas encore de SMA complète.
df_sma = df.dropna(subset=[sma_column]).copy()

num_higher = int((df_sma["Close"] > df_sma[sma_column]).sum())
num_lower_or_equal = int((df_sma["Close"] <= df_sma[sma_column]).sum())

print(f"Nombre de clôtures au-dessus de la SMA : {num_higher}")
print(f"Nombre de clôtures au-dessous ou égales à la SMA : {num_lower_or_equal}")
display(df_sma.tail())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df_sma.index, df_sma["Close"], label="Close ajusté", linewidth=1.2)
ax.plot(df_sma.index, df_sma[sma_column], label=sma_column, linewidth=1.5)
ax.set_title(f"{stock} — cours ajusté et moyenne mobile simple")
ax.set_xlabel("Date")
ax.set_ylabel("Prix")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## Variante : moyenne mobile exponentielle et signal RWB

La vidéo utilise aussi plusieurs EMA. La fonction suivante calcule ces EMA avec `pandas`, puis repère un signal lorsque toutes les EMA courtes sont au-dessus de toutes les EMA longues. Elle n’effectue pas d’achat réel et ne constitue pas une stratégie d’investissement.

In [ ]:
short_periods = [3, 5, 8, 10, 12, 15]
long_periods = [30, 35, 40, 45, 50, 60]

for period in short_periods + long_periods:
    df[f"EMA_{period}"] = df["Close"].ewm(span=period, adjust=False, min_periods=period).mean()

ema_columns = [f"EMA_{period}" for period in short_periods + long_periods]
rwb = df.dropna(subset=ema_columns).copy()

short_ema_max = rwb[[f"EMA_{p}" for p in short_periods]].max(axis=1)
long_ema_min = rwb[[f"EMA_{p}" for p in long_periods]].min(axis=1)
rwb["RWB_signal"] = short_ema_max > long_ema_min

print("Dernières lignes avec le signal RWB :")
display(rwb[["Close", "RWB_signal"] + ema_columns].tail())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(rwb.index, rwb["Close"], label="Close ajusté", color="black", linewidth=1.2)
for period in short_periods:
    ax.plot(rwb.index, rwb[f"EMA_{period}"], linewidth=0.8, alpha=0.8)
for period in long_periods:
    ax.plot(rwb.index, rwb[f"EMA_{period}"], linewidth=0.8, alpha=0.45, linestyle="--")

signal_days = rwb.index[rwb["RWB_signal"]]
ax.scatter(
    signal_days,
    rwb.loc[signal_days, "Close"],
    marker="^",
    color="green",
    s=30,
    label="Signal RWB",
)
ax.set_title(f"{stock} — EMA et signal RWB indicatif")
ax.set_xlabel("Date")
ax.set_ylabel("Prix")
ax.grid(True, alpha=0.25)
ax.legend(ncol=2)
fig.tight_layout()
plt.show()

## Adaptations principales

| Ancien code de la vidéo | Adaptation actuelle |
|---|---|
| `from pandas_datareader import data as pdr` | Supprimé |
| `yf.pdr_override()` | Supprimé |
| `pdr.get_data_yahoo(stock, start, now)` | `yf.download(stock, start=..., end=..., ...)` |
| `df["Adj Close"]` | `df["Close"]` avec `auto_adjust=True` |
| Accès `df["Close"][i]` | Opérations vectorisées pandas, plus robustes et plus rapides |
| Colonnes potentiellement multi-niveaux | `multi_level_index=False` et contrôle de sécurité |